In [202]:
import pandas as pd
import numpy  as np
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.metrics import confusion_matrix , accuracy_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


In [203]:
df = pd.read_csv("Titanic-Dataset.csv")

df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

df.drop(["PassengerId" , "Ticket" , "Name"] , axis = 1 , inplace=True)

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df = df.drop(['SibSp', 'Parch'], axis=1)


df['Age'] = df['Age'].fillna(df['Age'].mean())

df['Embarked'] = df.groupby('Pclass')['Embarked'].transform(lambda   x: x.fillna(x.mode()[0]))

df = df.drop("Cabin", axis=1)

df = pd.get_dummies(df , drop_first=True)

y = df['Survived']
X = df.drop(['Survived'] , axis = 1)


In [204]:

X_train , X_test , y_train , y_test = train_test_split(X , y ,test_size= 0.25 , random_state=42)


model = XGBClassifier(random_state=42 , eval_metric='logloss')

param_grid = {
    'max_depth': [3, 4, 5, 6],        
    'n_estimators': [50, 100, 150],     
    'learning_rate': [0.01, 0.05, 0.1], 
    'subsample': [0.7, 0.8, 1.0],     
    'colsample_bytree': [0.7, 0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)


grid_search.fit(X_train , y_train)

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print(confusion_matrix(y_test, y_pred))

print(accuracy_score(y_test, y_pred))



[[119  15]
 [ 21  68]]
0.8385650224215246
